# Урок 16 — Наслідування, MRO та Композиція на прикладі Titanic

---

> *«Жінки та діти — першими».*  
> Ця фраза, що пролунала на палубі Titanic у квітні 1912 року, — не просто морська традиція.  
> Це **алгоритм пріоритизації**. І сьогодні ми закодуємо його мовою Python.

---

## Мета уроку

У цьому уроці ми **не просто вивчаємо синтаксис наслідування**.

Ми хочемо зрозуміти три глибокі речі:

1. **Наслідування** — це механізм *пошуку методів*, а не копіювання коду.
2. **MRO (Method Resolution Order)** — Python завжди знає, *куди саме* звертатися за методом.
3. **Композиція** перемагає наслідування, коли мова йде про *поведінку системи*.

Titanic — ідеальна метафора: пасажири мають клас, стать, вік (структура — наслідування),  
але рятують їх **координатор і шлюпки** (поведінка — композиція).

---

## Зміст

| # | Розділ |
|---|--------|
| 1 | Вступ та мета |
| 2 | Завантаження та очищення даних |
| 3 | Базовий клас `Passenger` |
| 4 | Підкласи за класом каюти |
| 5 | Побудова списку пасажирів |
| 6 | MRO — алгоритм пошуку методів |
| 7 | Джек і Роуз — реальні персонажі |
| 8 | Аналітика реальних даних |
| 9 | Клас `Lifeboat` та наївна евакуація |
| 10 | Наслідування — не оркестрація |
| 11 | `RescueSystem` через композицію |
| 12 | Сценарій «всі врятовані» |
| 13 | Порівняльні графіки |
| 14 | Дебагінг: де зламана система? |
| 15 | Мінізавдання для студентів |
| 16 | Фінальне резюме |
| 17 | Питання для інтерв'ю |
| 18 | Висновок |


---

## Розділ 2 — Завантаження та очищення даних

Перш ніж будувати класи — нам потрібні дані.  
Завантажуємо датасет Titanic через `seaborn` (якщо встановлено) або напряму з GitHub.

Датасет містить 891 пасажира з такими полями:
- `survived` — 1 = вижив, 0 = загинув
- `pclass` — клас каюти (1, 2, 3)
- `sex` — стать (`male` / `female`)
- `age` — вік (з пропусками)
- `fare` — ціна квитка
- `embarked` — порт посадки (C, Q, S)
- `who` — хто це: `man`, `woman`, `child`
- `alone` — чи подорожує один


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

# Завантажуємо датасет: спочатку через seaborn, при невдачі — через URL
try:
    import seaborn as sns
    df_raw = sns.load_dataset("titanic")
    print("Завантажено через seaborn")
except Exception:
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
    df_raw = pd.read_csv(url)
    print("Завантажено через pandas/URL")

print(f"Розмір датасету: {df_raw.shape}")
df_raw.head()

In [ ]:
# Вибираємо потрібні колонки та очищаємо дані
cols = ['survived', 'pclass', 'sex', 'age', 'fare', 'embarked', 'who', 'alone']
df = df_raw[cols].copy()

# Заповнюємо пропуски медіанними/типовими значеннями
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df['embarked'] = df['embarked'].fillna('S')

# Додаємо унікальний ID кожному пасажиру
df['passenger_id'] = range(1, len(df) + 1)
df = df.reset_index(drop=True)

print(f"Чистий датасет: {df.shape}")
print(df.dtypes)
df.head(10)

---

## Розділ 3 — Базовий клас `Passenger`

### Що таке базовий клас?

**Базовий клас** (або *батьківський клас*, *суперклас*) — це клас, що описує **спільні властивості та поведінку** для цілої групи об'єктів.

У нашому випадку всі пасажири Titanic мають:
- ID, стать, вік, вартість квитка
- Статус виживання
- Клас каюти

### Відношення `is-a`

Наслідування виражає відношення **«є різновидом»** (`is-a`):

```
FirstClassPassenger  is-a  Passenger  ✓
ThirdClassPassenger  is-a  Passenger  ✓
Lifeboat             is-a  Passenger  ✗  ← це вже неправда!
```

Запам'ятайте: **якщо речення «X є різновидом Y» звучить природно — використовуй наслідування.  
Якщо ні — використовуй композицію.**

### Базовий клас: `Passenger`

Він описує *структуру* будь-якого пасажира.  
Метод `lifeboat_priority()` тут повертає `0` — це *заглушка*, яку підкласи перевизначать.


In [ ]:
class Passenger:
    """Базовий клас пасажира Titanic."""

    def __init__(self, passenger_id, sex, age, fare, survived, pclass, who, alone):
        self.passenger_id = passenger_id
        self.sex = sex
        self.age = float(age)
        self.fare = float(fare)
        self.survived = int(survived)
        self.pclass = int(pclass)
        self.who = who      # 'man', 'woman', 'child'
        self.alone = bool(alone)

    def is_child(self):
        """Повертає True, якщо вік менше 16 років."""
        return self.age < 16

    def survival_label(self):
        """Повертає читабельний статус виживання."""
        return "Врятований" if self.survived == 1 else "Загинув"

    def lifeboat_priority(self):
        # Базовий пріоритет — перевизначається у підкласах
        # Тут повертаємо 0: без класу — найнижчий пріоритет
        return 0

    def __repr__(self):
        status = "✓" if self.survived else "✗"
        return (f"[{status}] Passenger#{self.passenger_id} "
                f"pclass={self.pclass} {self.sex}/{self.age:.0f} "
                f"fare={self.fare:.1f}")

    @classmethod
    def from_series(cls, row):
        """Фабричний метод: створює об'єкт Passenger з рядка DataFrame."""
        return cls(
            passenger_id=row['passenger_id'],
            sex=row['sex'],
            age=row['age'],
            fare=row['fare'],
            survived=row['survived'],
            pclass=row['pclass'],
            who=row['who'],
            alone=row['alone']
        )


# Швидкий тест базового класу
test_passenger = Passenger(
    passenger_id=1, sex='male', age=22, fare=7.25,
    survived=0, pclass=3, who='man', alone=True
)
print(test_passenger)
print(f"Дитина? {test_passenger.is_child()}")
print(f"Статус: {test_passenger.survival_label()}")
print(f"Пріоритет: {test_passenger.lifeboat_priority()}")

---

## Розділ 4 — Підкласи за класом каюти

### Спеціалізація через наслідування

Тепер ми **спеціалізуємо** базовий клас.  
Кожен з трьох класів каюти — перший, другий, третій — **перевизначає** метод `lifeboat_priority()`.

Це класичний **override** (перевизначення):
- Python знаходить метод `lifeboat_priority()` у підкласі — і зупиняється тут.
- До батьківського класу він більше не йде (якщо ми явно не викликаємо `super()`).

### Логіка пріоритету

| Клас | База | +жінка | +дитина |
|------|------|--------|---------|
| 1-й  | 3    | +2     | +3      |
| 2-й  | 2    | +2     | +3      |
| 3-й  | 1    | +1     | +2      |

Тобто дитина першого класу — пріоритет 6 (найвищий),  
чоловік третього класу — пріоритет 1 (найнижчий).

> **Зверніть увагу**: `__init__` ми **не перевизначаємо** у підкласах,  
> бо всі поля однакові. Підклас автоматично успадковує `__init__` батька.


In [ ]:
class FirstClassPassenger(Passenger):
    """Пасажир першого класу — найвищий пріоритет рятування."""

    def lifeboat_priority(self):
        base = 3  # перший клас — базовий пріоритет 3
        if self.who == 'woman':
            base += 2
        if self.who == 'child':
            base += 3
        return base

    def __repr__(self):
        status = "✓" if self.survived else "✗"
        return (f"[{status}] 1st#{self.passenger_id} "
                f"{self.sex}/{self.age:.0f} "
                f"priority={self.lifeboat_priority()}")


class SecondClassPassenger(Passenger):
    """Пасажир другого класу — середній пріоритет рятування."""

    def lifeboat_priority(self):
        base = 2  # другий клас — базовий пріоритет 2
        if self.who == 'woman':
            base += 2
        if self.who == 'child':
            base += 3
        return base

    def __repr__(self):
        status = "✓" if self.survived else "✗"
        return (f"[{status}] 2nd#{self.passenger_id} "
                f"{self.sex}/{self.age:.0f} "
                f"priority={self.lifeboat_priority()}")


class ThirdClassPassenger(Passenger):
    """Пасажир третього класу — найнижчий базовий пріоритет."""

    def lifeboat_priority(self):
        base = 1  # третій клас — базовий пріоритет 1
        if self.who == 'woman':
            base += 1
        if self.who == 'child':
            base += 2
        return base

    def __repr__(self):
        status = "✓" if self.survived else "✗"
        return (f"[{status}] 3rd#{self.passenger_id} "
                f"{self.sex}/{self.age:.0f} "
                f"priority={self.lifeboat_priority()}")


# Словник: клас каюти → тип Python-класу
CLASS_MAP = {
    1: FirstClassPassenger,
    2: SecondClassPassenger,
    3: ThirdClassPassenger
}


def create_passenger(row):
    """Фабрична функція: вибирає правильний підклас за pclass."""
    cls = CLASS_MAP.get(int(row['pclass']), Passenger)
    return cls.from_series(row)


# Демонстрація різниці у repr та пріоритеті
p1 = FirstClassPassenger(1, 'female', 30, 100.0, 1, 1, 'woman', False)
p2 = SecondClassPassenger(2, 'male', 45, 50.0, 0, 2, 'man', True)
p3 = ThirdClassPassenger(3, 'male', 20, 7.25, 0, 3, 'man', True)

print("Порівняння підкласів:")
for p in [p1, p2, p3]:
    print(f"  {p}")

---

## Розділ 5 — Побудова списку пасажирів

Тепер застосуємо нашу ієрархію до реального датасету.

Функція `create_passenger()` автоматично вибирає правильний підклас на основі поля `pclass`.  
Це **фабричний патерн** — один із найпопулярніших патернів у ООП.

Беремо перші 100 рядків для демонстраційних цілей.


In [ ]:
# Беремо перші 100 пасажирів для демонстрації
passengers = [create_passenger(row) for _, row in df.head(100).iterrows()]

print(f"Створено об'єктів: {len(passengers)}")
print("\nПриклади repr:")
for p in passengers[:5]:
    print(p)

# Перевіряємо методи на першому пасажирі
print("\n--- Методи першого пасажира ---")
p = passengers[0]
print(f"is_child():         {p.is_child()}")
print(f"survival_label():   {p.survival_label()}")
print(f"lifeboat_priority(): {p.lifeboat_priority()}")
print(f"type:               {type(p).__name__}")

# Розподіл за типами класів
from collections import Counter
type_counts = Counter(type(p).__name__ for p in passengers)
print("\nРозподіл за підкласами:")
for cls_name, count in sorted(type_counts.items()):
    print(f"  {cls_name}: {count}")

---

## Розділ 6 — MRO: алгоритм пошуку методів

### Що таке MRO?

**MRO (Method Resolution Order)** — це **впорядкований список класів**, якими Python іде в пошуку методу чи атрибута.

Коли ви пишете `obj.some_method()`, Python **не копіює** методи у кожен клас.  
Він будує ланцюжок і **шукає по черзі**, поки не знайде потрібний метод.

### Ланцюжок для `FirstClassPassenger`:

```
FirstClassPassenger  →  Passenger  →  object
       ↑
  Тут є lifeboat_priority()  — знайшли, зупиняємося.
```

### Ланцюжок для `is_child()`:

```
FirstClassPassenger  →  Passenger  →  object
       ✗                    ↑
  Тут немає           Тут є is_child()  — знайшли!
```

### Алгоритм C3 Linearization

Python використовує алгоритм **C3 лінеаризації** для обчислення MRO.  
Він гарантує, що:
1. Дочірній клас завжди йде перед батьківським.
2. Порядок батьківських класів зберігається.
3. MRO є монотонним (клас не може з'явитись раніше, ніж його підкласи).

Перевіримо MRO програмно:


### MRO як маршрут, а не копіювання

Уявіть, що Python — це кур'єр, якому потрібно знайти метод `lifeboat_priority()`.
Він не копіює всі методи в один клас заздалегідь. Він **іде по маршруту** щоразу, коли ви робите виклик.

```
Виклик: jack.lifeboat_priority()

Маршрут пошуку (MRO):

  ThirdClassPassenger  ──▶  є lifeboat_priority()? ТАК → виконати, зупинитись
         │
    (якби не було)
         ▼
     Passenger         ──▶  є lifeboat_priority()? ТАК → виконати, зупинитись
         │
    (якби не було)
         ▼
      object           ──▶  є lifeboat_priority()? НІ → AttributeError
```

**Ключова ідея:**  
Метод у `ThirdClassPassenger` не замінює метод у `Passenger`.  
Він просто знаходиться **раніше в маршруті**, тому Python зупиняється на ньому першим.

Якщо прибрати `lifeboat_priority()` з `ThirdClassPassenger` — Python **автоматично** знайде версію з `Passenger`.  
Нічого не «зламається» і нічого не треба переписувати.

> **Саме тому наслідування — це механізм ПОШУКУ, а не копіювання.**


In [ ]:
# Переконаємось, що MRO — це справді лінійний маршрут
for cls in ThirdClassPassenger.mro():
    has_method = 'lifeboat_priority' in cls.__dict__
    marker = '← знайдено, зупиняємось' if has_method else '  (не знайдено, рухаємось далі)'
    print(f'  {cls.__name__:30s} {marker}')

print()
print('Результат виклику:', jack.lifeboat_priority() if 'jack' in dir() else '(jack ще не створений)')


In [ ]:
# Перегляд MRO для наших класів
print("MRO для FirstClassPassenger:")
for i, cls in enumerate(FirstClassPassenger.mro()):
    print(f"  {i+1}. {cls.__name__}")

print()
print("MRO для ThirdClassPassenger:")
for i, cls in enumerate(ThirdClassPassenger.mro()):
    print(f"  {i+1}. {cls.__name__}")

# Альтернативний спосіб — через __mro__
print()
print("__mro__ (кортеж):")
print(FirstClassPassenger.__mro__)

### Множинне наслідування та MRO

Найцікавіший сценарій — **множинне наслідування**.  
Коли клас `D` успадковує від `B` і `C`, які обидва успадковують від `A`,  
виникає **diamond problem** (ромбоподібна ієрархія).

```
        A
       / \
      B   C
       \ /
        D
```

Python вирішує це через C3 — клас `A` потрапляє в MRO лише **один раз**, найближче до кінця.

Розберемо приклад із `super()`:


In [ ]:
# Штучний приклад множинного наслідування (diamond problem)

class A:
    def say(self):
        print("A")

class B(A):
    def say(self):
        print("B")
        super().say()   # super() → наступний у MRO, не обов'язково A!

class C(A):
    def say(self):
        print("C")
        super().say()

class D(B, C):
    def say(self):
        print("D")
        super().say()

print("MRO для D:", [cls.__name__ for cls in D.mro()])
print("---")
print("Виклик D().say():")
D().say()

### Пояснення результату

MRO для `D`: `[D, B, C, A, object]`

Коли ми викликаємо `D().say()`, ланцюжок виглядає так:

```
D.say()  → виводить "D", потім super().say()
B.say()  → виводить "B", потім super().say()   ← super() у B — це C, не A!
C.say()  → виводить "C", потім super().say()
A.say()  → виводить "A"
```

Виведення: `D → B → C → A`

**Ключовий інсайт**: `super()` не означає «батьківський клас».  
`super()` означає «**наступний клас у MRO поточного об'єкта**».

Це чому `super()` у `B.say()` викликає `C.say()` — бо `C` наступний у MRO об'єкта `D`.

> **Висновок**: якщо всі класи у ланцюжку викликають `super()` — Python обходить кожен клас рівно один раз. Це кооперативне множинне наслідування.


---

## Розділ 7 — Джек і Роуз

Ніяка розповідь про Titanic не обходиться без двох головних персонажів.  
Давайте закодуємо їх явно — щоб потім відстежувати їхню долю в симуляціях.

- **Джек Доусон** — третій клас, 20 років, чоловік, один, квиток 7.25
- **Роуз ДеВітт Б'юкетер** — перший клас, 17 років, жінка, не сама, квиток 512

Зверніть увагу на різницю у пріоритеті — вона пояснює все.


In [ ]:
# Джек Доусон — третій клас, молодий чоловік
jack = ThirdClassPassenger(
    passenger_id=9999,
    sex='male',
    age=20,
    fare=7.25,
    survived=0,   # реальна доля
    pclass=3,
    who='man',
    alone=True
)

# Роуз ДеВітт Б'юкетер — перший клас
rose = FirstClassPassenger(
    passenger_id=9998,
    sex='female',
    age=17,
    fare=512.0,
    survived=1,
    pclass=1,
    who='woman',
    alone=False
)

print(jack)
print(rose)
print()
print(f'Пріоритет Джека:  {jack.lifeboat_priority()}  ← найнижчий')
print(f'Пріоритет Роуз:   {rose.lifeboat_priority()}  ← найвищий')
print()
print('--- Підказка до подальшого сюжету ---')
print('Джек МАЄ пріоритет (атрибут класу).')
print('Але хтось має цей пріоритет ПРОЧИТАТИ і ДІЯТИ на його основі.')
print('У першій моделі цього «когось» немає.')
print()
print(f'isinstance(jack, Passenger): {isinstance(jack, Passenger)}')
print(f'isinstance(rose, Passenger): {isinstance(rose, Passenger)}')


---

## Розділ 8 — Аналітика реальних даних

Перш ніж будувати симуляції — подивимось, що насправді сталося на Titanic.  
Реальні дані допоможуть нам зрозуміти, наскільки наша модель пріоритетів відповідає дійсності.

### Виживання за класом каюти

Перший клас рятувався значно краще — чому?  
Каюти 1-го класу знаходились ближче до палуби з шлюпками.  
Але і системна перевага — «жінки та діти першими» — також зіграла роль.


In [ ]:
# Графік 1: відсоток виживання за класом каюти
survival_by_class = df.groupby('pclass')['survived'].mean() * 100

fig, ax = plt.subplots()
bars = ax.bar(
    ['1 клас', '2 клас', '3 клас'],
    survival_by_class.values,
    color=['steelblue', 'cornflowerblue', 'lightsteelblue']
)
ax.set_title('Відсоток виживання за класом каюти (реальні дані Titanic)')
ax.set_ylabel('Виживання, %')
ax.set_ylim(0, 100)

# Підписуємо стовпці
for bar, val in zip(bars, survival_by_class.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val:.1f}%',
        ha='center', fontsize=12, fontweight='bold'
    )

plt.tight_layout()
plt.show()

print("Реальний відсоток виживання:")
for pclass, rate in survival_by_class.items():
    print(f"  Клас {pclass}: {rate:.1f}%")

In [ ]:
# Графік 2: виживання за статтю та класом
pivot = df.pivot_table(
    values='survived',
    index='pclass',
    columns='sex',
    aggfunc='mean'
) * 100

fig, ax = plt.subplots()
x = [0, 1, 2]
width = 0.35

ax.bar(
    [i - width / 2 for i in x],
    pivot['female'].values,
    width, label='Жінки', color='salmon'
)
ax.bar(
    [i + width / 2 for i in x],
    pivot['male'].values,
    width, label='Чоловіки', color='steelblue'
)

ax.set_title('Виживання за статтю та класом (реальні дані Titanic)')
ax.set_xticks(x)
ax.set_xticklabels(['1 клас', '2 клас', '3 клас'])
ax.set_ylabel('Виживання, %')
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.show()

print("Деталі (жінки/чоловіки):")
print(pivot.round(1).to_string())

### Спостереження з реальних даних

| Факт | Висновок |
|------|----------|
| Жінки 1-го класу: ~97% вижили | Стать + клас — домінуючий фактор |
| Чоловіки 3-го класу: ~13% | Найважча доля |
| Жінки 2-го і 3-го класу: >50% | «Жінки першими» спрацювало |
| Чоловіки 1-го класу: ~37% | Навіть багатство не рятувало |

Наша модель пріоритетів (жінка +2, дитина +3, 1-й клас +3) — досить добре відображає цей патерн.


---

## Розділ 9 — Клас `Lifeboat` та наївна евакуація

### Клас `Lifeboat`

Тепер нам потрібна **шлюпка**.  
Це проста структура: є місткість, є список пасажирів, є метод `add_passenger()`.

### Перша модель — без координатора

Що станеться, якщо просто брати пасажирів у порядку, в якому вони з'являються,  
і садити в шлюпки без будь-якого пріоритету?

Це **наївна модель** — без оркестрації.  
Перші ліпші 30 пасажирів займуть усі місця, решта загине.  
Джек, зрозуміло, потрапить наприкінці черги...


In [ ]:
class Lifeboat:
    """Рятувальна шлюпка з обмеженою місткістю."""

    def __init__(self, boat_id, capacity=20):
        self.boat_id = boat_id
        self.capacity = capacity
        self.passengers = []

    def has_space(self):
        """Перевіряє, чи є вільні місця."""
        return len(self.passengers) < self.capacity

    def add_passenger(self, passenger):
        """Додає пасажира, якщо є місце. Повертає True при успіху."""
        if self.has_space():
            self.passengers.append(passenger)
            return True
        return False

    def __repr__(self):
        return f"Lifeboat#{self.boat_id} [{len(self.passengers)}/{self.capacity}]"


# Тест шлюпки
test_boat = Lifeboat(1, capacity=3)
for p in passengers[:4]:
    result = test_boat.add_passenger(p)
    print(f"Додано {p.passenger_id}? {result} — {test_boat}")

In [ ]:
# Наївна модель: немає координатора
# Пасажири садяться у тому порядку, в якому вони є у списку — без жодного пріоритету

demo_passengers = passengers[:50] + [jack]   # Джек — останній у черзі

naive_boats = [Lifeboat(i, capacity=10) for i in range(1, 4)]  # 3 шлюпки × 10 = 30 місць на 51 людину

saved_naive = []
not_saved_naive = []

for passenger in demo_passengers:
    boarded = False
    for boat in naive_boats:
        if boat.add_passenger(passenger):
            saved_naive.append(passenger)
            boarded = True
            break
    if not boarded:
        not_saved_naive.append(passenger)

print('=' * 45)
print('  НАЇВНА МОДЕЛЬ — без координатора')
print('=' * 45)
print(f'Всього пасажирів:  {len(demo_passengers)}')
print(f'Місць у шлюпках:   {sum(b.capacity for b in naive_boats)}')
print(f'Врятовано:         {len(saved_naive)}')
print(f'Не врятовано:      {len(not_saved_naive)}')

# ── Доля Джека ──────────────────────────────────────────────────────────────
jack_saved_naive = jack in saved_naive
print()
print('>>> Доля Джека Доусона:')
if jack_saved_naive:
    print('    Джек ВРЯТОВАНИЙ (йому пощастило зайти раніше)')
else:
    print('    Джек ЗАГИНУВ — місця скінчились до того, як дійшла черга')
    print('    Хоча lifeboat_priority() у нього є. Просто ніхто його не використав.')

# ── Кого не врятували — по класах ───────────────────────────────────────────
not_saved_classes = Counter(p.pclass for p in not_saved_naive)
print()
print('Не врятовані по класах:', dict(not_saved_classes))
print()
print('Стан шлюпок:')
for boat in naive_boats:
    print(f'  {boat}')
print()
print('>>> Шлюпки ПОВНІ, але третій клас постраждав більше за інших.')
print('>>> Причина — не брак класів. Причина — відсутність координатора.')


---

## Реальний Titanic: 3-й клас масово гинув

Перш ніж пояснювати «чому система зламалась» — подивіться на реальні дані.  
Наша наївна модель не вигадала несправедливість. Вона **відтворила** її.

Наступний графік — це не наша модель. Це справжній Titanic, 1912 рік.


In [ ]:
# Реальний Titanic: відсоток виживання за класом каюти
# Дані з датасету — не з нашої симуляції
survival_real = df.groupby('pclass')['survived'].mean() * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = survival_real.plot(kind='bar', ax=ax, color=['steelblue', 'steelblue', 'tomato'], rot=0)
ax.set_title('Реальний Titanic: виживання за класом каюти', fontsize=13)
ax.set_xlabel('Клас каюти')
ax.set_ylabel('Виживання, %')
ax.set_xticklabels(['1 клас', '2 клас', '3 клас'])
ax.set_ylim(0, 100)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.5,
            f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

# Кількість для контексту
counts = df.groupby('pclass').agg(
    всього=('survived','count'),
    врятованих=('survived','sum')
).rename(index={1:'1 клас', 2:'2 клас', 3:'3 клас'})
counts['загинуло'] = counts['всього'] - counts['врятованих']
print(counts)


**Висновок з реальних даних:**

| Клас | Виживання |
|------|-----------|
| 1-й  | ~63%      |
| 2-й  | ~47%      |
| 3-й  | ~24%      |

Це не вигадка. Третій клас мав нижчий доступ до шлюпок —  
бо **не було системи**, яка гарантувала б рівний доступ.

Наша наївна модель відтворила саме цю ситуацію:  
без координатора перемагає той, хто першим опинився у черзі.

> **Архітектурна проблема** — це не технічний баг.  
> Це відсутність компонента, який відповідає за справедливий розподіл.


---

## Debug: чому Джек не врятувався?

Давайте зіграємо в детективів.  
Замість того щоб одразу читати відповідь — простежимо за системою крок за кроком.

**Гіпотеза:**  
Джек загинув через одну з трьох причин:
- (А) Шлюпки вже були повні, коли дійшла його черга
- (Б) Логіка `add_passenger()` має баг
- (В) Пріоритет Джека не читається ніде в коді

Запустіть наступну клітинку — і знайдіть правильну відповідь.


In [ ]:
# Крок 1: перевіряємо місткість шлюпок у наївній моделі
total_capacity = sum(b.capacity for b in naive_boats)
print(f'Місць у шлюпках: {total_capacity}')
print(f'Пасажирів у черзі: {len(demo_passengers)}')
print(f'Дефіцит місць: {len(demo_passengers) - total_capacity}')
print()

# Крок 2: хто зайняв останні місця?
print('Останні 5 пасажирів, що потрапили у шлюпки:')
for p in saved_naive[-5:]:
    print(f'  {p}  priority={p.lifeboat_priority()}')
print()

# Крок 3: де Джек у черзі?
jack_position = demo_passengers.index(jack)
print(f'Позиція Джека у черзі: {jack_position + 1} з {len(demo_passengers)}')
print(f'Місць вистачало для перших {total_capacity} пасажирів.')
print()

# Крок 4: чи читається priority у наївному циклі?
print('Перевіримо наївний цикл:')
print('  for passenger in demo_passengers:  ← порядок: як у списку, НЕ за priority')
print('      boat.add_passenger(passenger)  ← жодного сортування!')
print()
print(f'Пріоритет Джека: {jack.lifeboat_priority()} — але ніхто його не читав.')
print()
print('Відповідь: (А) + (В) — місць не вистачило І пріоритет не враховувався.')
print('Причина: відсутній координатор, який би сортував перед посадкою.')


---

## Чому система зламалась?

Давайте чесно відповімо на це питання — перед тим як рухатись далі.

---

### Що у нас є

- `ThirdClassPassenger` — **є** пасажиром (наслідування `is-a`) ✓  
- `lifeboat_priority()` — **повертає** `1` для Джека ✓  
- `Lifeboat.add_passenger()` — **може** прийняти пасажира ✓  

Усе є. І все одно — Джек загинув.

---

### Чого не вистачає

```
Наявне:                          Відсутнє:
──────────────────────           ──────────────────────────────────────
Jack.lifeboat_priority() → 1    ← Хтось, хто читає цей пріоритет
Lifeboat.add_passenger()         ← Хтось, хто вирішує черговість
ThirdClassPassenger is Passenger ← Хтось, хто керує всією евакуацією
```

**Наслідування описує СТРУКТУРУ — хто ким є.**  
Воно не описує ПОВЕДІНКУ СИСТЕМИ — хто ким керує.

---

### Де конкретно зламано

```python
# Ось цей цикл — наш «координатор»
for passenger in demo_passengers:   # ← порядок не враховує пріоритет!
    for boat in naive_boats:
        if boat.add_passenger(passenger):
            break
```

Цикл бере пасажирів у тому порядку, в якому вони опинились у списку.  
Він **ніколи не читає** `lifeboat_priority()`.  
Пасажири першого класу зайняли місця просто тому, що були на початку списку.

---

### Принцип, якого не вистачало

> *«Favour object composition over class inheritance»*  
> — Gang of Four, 1994

Ми потребуємо **окремого об'єкта-координатора**, який:
1. знає про всіх пасажирів,
2. знає про всі шлюпки,
3. **сам містить** логіку розподілу.

Це і є **RescueSystem** — не новий підклас `Passenger`, а **композиція**.


---

## Розділ 10 — Наслідування — це не оркестрація

### Найважливіший урок цього заняття

Подивіться на наш клас `ThirdClassPassenger`.  
У нього є метод `lifeboat_priority()` — який повертає `1` для чоловіка.

Але що з цим числом **хтось зробив**?  
У наївній моделі — **нічого**. Ніхто не читав це значення при посадці.

**Ось де наслідування закінчується і починається проблема.**

---

### Що описує наслідування?

Наслідування описує **структуру**: хто ким є (`is-a`).

```
FirstClassPassenger  is-a  Passenger  ✓
```

Це статична характеристика об'єкта.  
`lifeboat_priority()` у підкласі — це просто **властивість** об'єкта, не поведінка системи.

---

### Що наслідування **не може** зробити?

Воно **не може** описати:
- Як пасажири впорядковуються перед шлюпками
- Хто вирішує, яка шлюпка приймає якого пасажира
- Коли зупинити евакуацію

Це **системна поведінка** — і нею повинен керувати **координатор**.

---

### Принцип Gang of Four (GoF)

> *«Favour object composition over class inheritance.»*  
> — Design Patterns: Elements of Reusable Object-Oriented Software, 1994

Перекладається як:
> *«Надавайте перевагу композиції об'єктів над наслідуванням класів.»*

Чому?  
Тому що наслідування — **жорстке**: змінити ієрархію класів важко без зламу коду.  
Композиція — **гнучка**: ви замінюєте компоненти без зміни структури.

---

### Аналогія

```
Наслідування = опис ДНК пасажира
               (хто він є, що він може)

Композиція   = координатор евакуації
               (читає властивості пасажирів і ухвалює рішення)
```

Без координатора пасажири можуть знати свій пріоритет,  
але це нічого не змінює — якщо ніхто не діє на основі цих знань.


---

## Розділ 11 — `RescueSystem` через композицію

### Відношення `has-a` замість `is-a`

**Композиція** — це відношення «**має**» (`has-a`):

```
RescueSystem  has-a  list[Lifeboat]   ✓
RescueSystem  has-a  list[Passenger]  ✓
RescueSystem  is-a   Lifeboat         ✗  ← неправда!
```

`RescueSystem` **не наслідує** ні `Lifeboat`, ні `Passenger`.  
Він **містить** їх і **керує** ними.

### Ключова відмінність від наївної моделі

У методі `evacuate()` ми спочатку **сортуємо пасажирів за пріоритетом**,  
і тільки потім розсаджуємо по шлюпках.  
Тепер `lifeboat_priority()` реально впливає на порятунок!


In [ ]:
class RescueSystem:
    """
    Координатор евакуації.

    Він не наслідує ні Lifeboat, ні Passenger.
    Він МАЄ їх (композиція) і керує ними.

    has-a:
      self.lifeboats  — список шлюпок
      self.saved      — врятовані пасажири
      self.not_saved  — не врятовані пасажири
    """

    def __init__(self, lifeboats):
        self.lifeboats = lifeboats    # has-a: список шлюпок
        self.saved = []
        self.not_saved = []

    def evacuate(self, passengers):
        """Евакуює пасажирів за спадаючим пріоритетом."""
        # Сортуємо за спаданням пріоритету
        # Ось тут lifeboat_priority() реально використовується!
        ordered = sorted(passengers, key=lambda p: p.lifeboat_priority(), reverse=True)

        for passenger in ordered:
            boarded = False
            for boat in self.lifeboats:
                if boat.has_space():
                    boat.add_passenger(passenger)
                    self.saved.append(passenger)
                    boarded = True
                    break
            if not boarded:
                self.not_saved.append(passenger)

        return self.saved, self.not_saved

    def report(self):
        """Виводить звіт евакуації."""
        total = len(self.saved) + len(self.not_saved)
        print("=== Звіт RescueSystem ===")
        print(f"Всього пасажирів: {total}")
        print(f"Врятовано:        {len(self.saved)} ({len(self.saved)/total*100:.1f}%)")
        print(f"Не врятовано:     {len(self.not_saved)} ({len(self.not_saved)/total*100:.1f}%)")

        saved_classes = Counter(p.pclass for p in self.saved)
        not_saved_classes = Counter(p.pclass for p in self.not_saved)
        print(f"\nВрятовані по класах:     {dict(sorted(saved_classes.items()))}")
        print(f"Не врятовані по класах:  {dict(sorted(not_saved_classes.items()))}")


print("Клас RescueSystem визначено.")

In [ ]:
# Запускаємо RescueSystem — та сама кількість шлюпок, ті самі пасажири
rescue_boats = [Lifeboat(i, capacity=10) for i in range(1, 4)]   # 3 шлюпки × 10 = 30 місць
demo_p2 = passengers[:50] + [jack, rose]                          # + Джек і Роуз

system = RescueSystem(rescue_boats)
saved, not_saved = system.evacuate(demo_p2)

print('=' * 45)
print('  RESCUE SYSTEM — з координатором')
print('=' * 45)
system.report()

print()
jack_saved_v2 = jack in saved
rose_saved_v2 = rose in saved
print(f'>>> Джек врятований? {"ТАК!" if jack_saved_v2 else "НІ"}')
print(f'>>> Роуз врятована?  {"ТАК!" if rose_saved_v2 else "НІ"}')
print()
print('Що змінилось від наївної моделі?')
print(f'  Наївна:       врятовано {len(saved_naive)}/{len(demo_passengers)}')
print(f'  RescueSystem: врятовано {len(saved)}/{len(demo_p2)}')
print()
print('Кількість шлюпок — однакова.')
print('Кількість місць  — однакова.')
print('Змінився тільки КООРДИНАТОР — і результат інший.')


In [ ]:
# Будуємо DataFrame результатів RescueSystem для аналізу
rescue_results = pd.DataFrame({
    'passenger_id': [p.passenger_id for p in demo_p2],
    'pclass':       [p.pclass       for p in demo_p2],
    'who':          [p.who          for p in demo_p2],
    'priority':     [p.lifeboat_priority() for p in demo_p2],
    'saved':        [1 if p in saved else 0 for p in demo_p2],
})

# Також будуємо DataFrame наївної моделі для порівняння
naive_results = pd.DataFrame({
    'pclass':   [p.pclass for p in demo_passengers],
    'saved':    [1 if p in saved_naive else 0 for p in demo_passengers],
})

# Порівняльний графік: % врятованих по класах у двох моделях
naive_rate  = naive_results.groupby('pclass')['saved'].mean() * 100
rescue_rate = rescue_results.groupby('pclass')['saved'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
x = [0, 1, 2]
w = 0.35
ax.bar([i - w/2 for i in x], naive_rate.reindex([1,2,3], fill_value=0),  w, label='Наївна модель', color='tomato')
ax.bar([i + w/2 for i in x], rescue_rate.reindex([1,2,3], fill_value=0), w, label='RescueSystem',  color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(['1 клас', '2 клас', '3 клас'])
ax.set_title('До vs Після: % врятованих за класом (однакова кількість шлюпок)', fontsize=12)
ax.set_ylabel('% врятованих у симуляції')
ax.set_ylim(0, 105)
ax.legend()
plt.tight_layout()
plt.show()

print('Зведена таблиця (RescueSystem):')
print(rescue_results.groupby('pclass').agg(
    всього=('saved','count'),
    врятовано=('saved','sum'),
    відсоток=('saved','mean')
).round(2))


### Ми змінили не людей — ми змінили систему

Порівняйте дві моделі:

| | Наївна модель | RescueSystem |
|---|---|---|
| Пасажири | ті самі | ті самі |
| Класи Python | ті самі | ті самі |
| Шлюпки | 3 × capacity=10 | 3 × capacity=10 |
| Що змінилось | — | з'явився координатор |
| Результат | несправедливий розподіл | пріоритет впливає на порятунок |

**Один новий клас** — `RescueSystem` — змінив поведінку всієї системи.  
Не тому що він «кращий клас». А тому що він **взяв на себе відповідальність**  
за координацію між об'єктами.

> Це і є суть композиції: не будувати кращі окремі об'єкти —  
> а будувати правильні **відносини між об'єктами**.


---

## Контраст: До і Після

| | Наївна модель | RescueSystem |
|---|---|---|
| Порядок посадки | За порядком у списку | За `lifeboat_priority()` |
| Хто вирішує? | Ніхто — просто цикл | Координатор `evacuate()` |
| Пріоритет враховується? | НІ | ТАК |
| Третій клас | Страждає найбільше | Отримує місця за пріоритетом |
| Де логіка системи? | Розпорошена по циклах | Інкапсульована у `RescueSystem` |

---

### Чому це важливо для архітектора?

```
inheritance-only модель:
  Passenger → знає, хто він
  Lifeboat  → знає, скільки в ній місць
  [цикл]    → робить «щось» без логіки
  ──────────────────────────────────────
  Система не існує. Є набір ізольованих класів.

Модель з композицією:
  Passenger    → знає, хто він (структура)
  Lifeboat     → знає, скільки в ній місць (структура)
  RescueSystem → has-a [lifeboats], has-a [passengers]
               → evacuate() — системна поведінка
  ──────────────────────────────────────────────────────
  Система існує. Поведінка інкапсульована у координаторі.
```


In [ ]:
# Порівняльний графік: виживання по класах у двох моделях
import matplotlib.pyplot as plt
from collections import Counter

# --- Наївна модель ---
naive_saved_by_class   = Counter(p.pclass for p in saved_naive)
naive_notsaved_by_class = Counter(p.pclass for p in not_saved_naive)

# --- RescueSystem ---
rescue_saved_by_class   = Counter(p.pclass for p in saved)
rescue_notsaved_by_class = Counter(p.pclass for p in not_saved)

classes = [1, 2, 3]
labels = ['1 клас', '2 клас', '3 клас']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ── Наївна ──
ns = [naive_saved_by_class.get(c, 0) for c in classes]
nn = [naive_notsaved_by_class.get(c, 0) for c in classes]
ax1.bar(labels, ns, label='Врятовані')
ax1.bar(labels, nn, bottom=ns, label='Не врятовані', color='tomato')
ax1.set_title('До: Наївна модель (без координатора)')
ax1.set_ylabel('Кількість пасажирів')
ax1.legend()

# ── RescueSystem ──
rs = [rescue_saved_by_class.get(c, 0) for c in classes]
rn = [rescue_notsaved_by_class.get(c, 0) for c in classes]
ax2.bar(labels, rs, label='Врятовані')
ax2.bar(labels, rn, bottom=rs, label='Не врятовані', color='tomato')
ax2.set_title('Після: RescueSystem (з координатором)')
ax2.set_ylabel('Кількість пасажирів')
ax2.legend()

plt.suptitle('Порівняння моделей евакуації — однакова кількість шлюпок і місць', fontsize=13)
plt.tight_layout()
plt.show()

print('Висновок: при однаковій місткості шлюпок координатор дає кращий розподіл по класах.')


---

## Розділ 12 — Сценарій «Всі врятовані»

А що якби на Titanic було **достатньо шлюпок**?  
Historyфакт: на кораблі було місце лише для **1178 осіб** з 2224 на борту.  
Шлюпок вистачило б на половину.

Побудуємо сценарій, де шлюпок достатньо для **всіх** — і перевіримо, чи врятується Джек.


In [ ]:
import math
from IPython.display import Image, display


def calculate_lifeboats_needed(passengers, capacity_per_boat=20):
    """Розраховує кількість шлюпок для врятування всіх."""
    return math.ceil(len(passengers) / capacity_per_boat)


# Будуємо список з усіх пасажирів датасету + Джек і Роуз
all_passengers = [create_passenger(row) for _, row in df.iterrows()] + [jack, rose]

n_boats = calculate_lifeboats_needed(all_passengers, capacity_per_boat=30)
print(f"Пасажирів разом:  {len(all_passengers)}")
print(f"Потрібно шлюпок (по 30 місць): {n_boats}")

# Запускаємо систему з достатньою кількістю шлюпок
enough_boats = [Lifeboat(i, capacity=30) for i in range(1, n_boats + 1)]
full_system = RescueSystem(enough_boats)
saved_all, not_saved_all = full_system.evacuate(all_passengers)

full_system.report()
print()
jack_saved = jack in saved_all
print(f"Джек врятований? {'Так!' if jack_saved else 'Ні'}")
print(f"Роуз врятована?  {'Так!' if rose in saved_all else 'Ні'}")

if jack_saved:
    display(Image(filename="images.jpg"))

---

## Розділ 13 — Порівняльні графіки

Відвізуалізуємо різницю між наївною моделлю та `RescueSystem`.  
Також подивимось, як пріоритезація вплинула на розподіл виживання по класах.


In [ ]:
# Графік: наївна модель vs RescueSystem
demo_size = len(demo_passengers)
naive_rate = len(saved_naive) / demo_size * 100
rescue_rate = len(saved) / len(demo_p2) * 100

fig, ax = plt.subplots()
bars = ax.bar(
    ['Наївна модель\n(без координатора)', 'RescueSystem\n(з координатором)'],
    [naive_rate, rescue_rate],
    color=['tomato', 'steelblue'],
    width=0.5
)
ax.set_title('Порівняння моделей евакуації (однакова кількість шлюпок)')
ax.set_ylabel('% врятованих від загальної кількості пасажирів')
ax.set_ylim(0, 100)

for bar, val in zip(bars, [naive_rate, rescue_rate]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val:.1f}%',
        ha='center', fontsize=13, fontweight='bold'
    )

plt.tight_layout()
plt.show()

print(f"Наївна модель:   {naive_rate:.1f}% ({len(saved_naive)}/{demo_size})")
print(f"RescueSystem:    {rescue_rate:.1f}% ({len(saved)}/{len(demo_p2)})")
print()
print("Примітка: обидві моделі врятовують однакову кількість осіб,")
print("але RescueSystem рятує пасажирів з ВИЩИМ пріоритетом.")

In [ ]:
# Графік: розподіл врятованих/не врятованих по класах (RescueSystem)
saved_c = Counter(p.pclass for p in saved)
not_saved_c = Counter(p.pclass for p in not_saved)

classes = [1, 2, 3]
saved_vals = [saved_c.get(c, 0) for c in classes]
not_saved_vals = [not_saved_c.get(c, 0) for c in classes]

fig, ax = plt.subplots()
x = range(3)

ax.bar(x, saved_vals, label='Врятовані', color='steelblue')
ax.bar(x, not_saved_vals, bottom=saved_vals, label='Не врятовані', color='tomato')

ax.set_xticks(list(x))
ax.set_xticklabels(['1 клас', '2 клас', '3 клас'])
ax.set_title('Розподіл врятованих/не врятованих по класах (RescueSystem)')
ax.set_ylabel('Кількість пасажирів')
ax.legend()
plt.tight_layout()
plt.show()

print("Деталі:")
for c in classes:
    s = saved_c.get(c, 0)
    ns = not_saved_c.get(c, 0)
    total_c = s + ns
    rate = s / total_c * 100 if total_c > 0 else 0
    print(f"  Клас {c}: врятовано {s}/{total_c} ({rate:.0f}%)")

---

## Розділ 14 — Дебагінг: де зламана система?

### Три запитання для розслідування

Розберемо покроково, чому наївна модель дала поганий результат.

---

**Питання 1:**  
Чому Джек не потрапив у шлюпку — хоча `lifeboat_priority()` у нього є?

**Питання 2:**  
Хто з пасажирів зайняв місця першим — і чому?

**Питання 3:**  
Який клас у нашій системі *повинен* нести відповідальність за логіку евакуації?

---

> Перш ніж бачити відповіді у наступних клітинках — спробуйте відповісти самостійно.


In [ ]:
# Відповідь на питання 1: чому Джек не потрапив у наївній моделі?

print('=== Аналіз наївної черги ===' )
print('Перші 6 пасажирів у demo_passengers (ті, хто зайняв місця):')
for p in demo_passengers[:6]:
    print(f'  {p}  priority={p.lifeboat_priority()}')
print('  ...')
print(f'Джек (останній у черзі): {jack}  priority={jack.lifeboat_priority()}')
print()
print('Висновок: наївна модель ніколи не читала lifeboat_priority().')
print('Пріоритет — атрибут об\'єкта, але ніхто не діяв на його основі.')
print()

# Відповідь на питання 2: хто зайняв місця?
first_class_in_naive = sum(1 for p in saved_naive if p.pclass == 1)
third_class_in_naive = sum(1 for p in saved_naive if p.pclass == 3)
print(f'Врятованих 1-го класу у наївній: {first_class_in_naive}')
print(f'Врятованих 3-го класу у наївній: {third_class_in_naive}')
print()
first_class_in_rescue = sum(1 for p in saved if p.pclass == 1)
third_class_in_rescue = sum(1 for p in saved if p.pclass == 3)
print(f'Врятованих 1-го класу у RescueSystem: {first_class_in_rescue}')
print(f'Врятованих 3-го класу у RescueSystem: {third_class_in_rescue}')


In [ ]:
# Питання 2: яка шлюпка взяла Джека у RescueSystem?
found_in_boat = None
for boat in rescue_boats:
    if jack in boat.passengers:
        found_in_boat = boat
        break

if found_in_boat:
    print(f"Джек у: {found_in_boat}")
    print(f"Пасажири у цій шлюпці:")
    for p in found_in_boat.passengers:
        print(f"  {p}  priority={p.lifeboat_priority()}")
else:
    print("Джек не у жодній шлюпці")

print()
print("Стан усіх шлюпок після RescueSystem:")
for boat in rescue_boats:
    print(f"  {boat}")

In [ ]:
# Питання 3: перевіримо правильність пріоритетів
# Виведемо топ-10 і боттом-10 за пріоритетом

sorted_by_priority = sorted(demo_p2, key=lambda p: p.lifeboat_priority(), reverse=True)

print("ТОП-10 (найвищий пріоритет — перші у шлюпки):")
for p in sorted_by_priority[:10]:
    print(f"  priority={p.lifeboat_priority()}  {p}")

print()
print("БОТТОМ-5 (найнижчий пріоритет):")
for p in sorted_by_priority[-5:]:
    print(f"  priority={p.lifeboat_priority()}  {p}")

---

## Розділ 15 — Мінізавдання для студентів

Спробуйте виконати ці завдання самостійно. Вони підсилюють розуміння матеріалу.

### Завдання 1
Змініть місткість шлюпок з 10 на 15 і запустіть `RescueSystem` знову.  
Чи врятується Джек тепер?

### Завдання 2
Додайте метод `is_priority_boarding()` до базового класу `Passenger`,  
який повертає `True` якщо `who == 'woman'` або `who == 'child'`.

### Завдання 3
Змініть логіку пріоритету: якщо вік < 10 — найвищий пріоритет (наприклад, +5).  
Перевизначте у підкласах або у базовому класі.

### Завдання 4 (теоретичне)
Що станеться, якщо два класи у множинному наслідуванні **не викликають `super()`**?  
Перевірте на прикладі B і C з розділу 6.


In [ ]:
# Завдання 1: більші шлюпки
bigger_boats = [Lifeboat(i, capacity=15) for i in range(1, 4)]   # 15 місць замість 10
system_bigger = RescueSystem(bigger_boats)
saved_bigger, not_saved_bigger = system_bigger.evacuate(demo_p2)
system_bigger.report()
print(f"\nДжек врятований? {'Так!' if jack in saved_bigger else 'Ні'}")

In [ ]:
# Завдання 2: TODO — додайте метод is_priority_boarding() до Passenger

# TODO: розкоментуйте і доповніть код нижче

# def is_priority_boarding(self):
#     """Повертає True, якщо пасажир має пріоритетну посадку."""
#     return ___  # ваш код тут

# Passenger.is_priority_boarding = is_priority_boarding   # monkey-patch для демонстрації

# Після додавання — протестуйте:
# print(jack.is_priority_boarding())   # очікуємо False
# print(rose.is_priority_boarding())   # очікуємо True

print("Завдання 2: реалізуйте is_priority_boarding() і розкоментуйте код")

In [ ]:
# Завдання 3: TODO — змінити пріоритет для дітей до 10 років

# TODO: створіть нову версію ThirdClassPassenger з іншою логікою

# class ThirdClassPassengerV2(Passenger):
#     def lifeboat_priority(self):
#         if self.age < 10:         # маленькі діти — найвищий пріоритет
#             return ___
#         base = 1
#         if self.who == 'woman': base += 1
#         if self.who == 'child': base += 2
#         return base

# Створіть маленьку дитину і перевірте пріоритет:
# baby = ThirdClassPassengerV2(9997, 'male', 5, 7.25, 0, 3, 'child', False)
# print(f"Пріоритет малюка: {baby.lifeboat_priority()}")

print("Завдання 3: реалізуйте ThirdClassPassengerV2 з підвищеним пріоритетом для age < 10")

In [ ]:
# Завдання 4: що станеться без super() у множинному наслідуванні?

class A2:
    def say(self):
        print("A2")

class B2(A2):
    def say(self):
        print("B2")
        # Навмисно НЕ викликаємо super()

class C2(A2):
    def say(self):
        print("C2")
        # Навмисно НЕ викликаємо super()

class D2(B2, C2):
    def say(self):
        print("D2")
        super().say()

print("MRO для D2:", [cls.__name__ for cls in D2.mro()])
print("---")
print("Виклик D2().say():")
D2().say()
print()
print("Висновок: B2.say() не викликає super() → C2 і A2 НЕ виконуються.")
print("Ланцюжок кооперативного наслідування розривається!")

---

## Killer Challenge: справедлива евакуація

### Постановка задачі

Поточний `RescueSystem` дає перевагу першому класу — бо там вище базовий priority.  
У результаті пасажири 3-го класу все одно отримують менше місць.

**Твоя задача:** змінити логіку так, щоб:
- жінки та діти мали абсолютний пріоритет (незалежно від класу каюти),
- але третій клас **не ігнорувався повністю** — він отримував пропорційну частку місць.

Це не проста задача. Вона потребує думати про **систему**, а не про окремий клас.

---

### Підказки (не підглядай одразу)

**Підказка 1 — де змінювати:**  
Не треба переписувати `Passenger` або `Lifeboat`.  
Логіка розподілу живе в `RescueSystem.evacuate()` — тільки там.

**Підказка 2 — ідея з квотами:**  
Що якби кожна шлюпка мала «квоту» для кожного класу?  
Наприклад: min 2 місця для третього класу у кожній шлюпці.

**Підказка 3 — два проходи:**  
Евакуацію можна розбити на два проходи:  
1. Спочатку — жінки та діти (усіх класів).  
2. Потім — всі решта, хто залишився.

---

**Критерій успіху:**  
Після вашого `FairRescueSystem.evacuate(demo_p2)` виведіть відсоток врятованих  
по кожному класу — і переконайтеся, що 3-й клас не отримав 0%.


In [ ]:
# Killer Challenge: FairRescueSystem
# Завдання: зробити евакуацію більш справедливою
# Правила: жінки та діти — абсолютний пріоритет,
#           але 3-й клас не ігнорується.

class FairRescueSystem:
    def __init__(self, lifeboats):
        self.lifeboats = lifeboats
        self.saved = []
        self.not_saved = []

    def evacuate(self, passengers):
        # TODO: реалізуйте справедливу логіку евакуації
        # Підказка: розбийте пасажирів на групи і обробляйте їх окремо

        priority_group = []  # TODO: жінки та діти
        others = []           # TODO: всі решта

        for p in passengers:
            pass  # TODO: розподіліть пасажирів по групах

        all_ordered = priority_group + others  # простий варіант: два проходи

        for passenger in all_ordered:
            boarded = False
            for boat in self.lifeboats:
                if boat.has_space():
                    boat.add_passenger(passenger)
                    self.saved.append(passenger)
                    boarded = True
                    break
            if not boarded:
                self.not_saved.append(passenger)

        return self.saved, self.not_saved

    def report_by_class(self):
        from collections import Counter
        saved_c = Counter(p.pclass for p in self.saved)
        total_c = Counter(p.pclass for p in self.saved + self.not_saved)
        print('=== FairRescueSystem — результат ===')
        for c in [1, 2, 3]:
            s = saved_c.get(c, 0)
            t = total_c.get(c, 0)
            pct = s / t * 100 if t else 0
            print(f'  {c} клас: {s}/{t} ({pct:.1f}%)')


# Тест (запустіть після реалізації)
# fair_boats = [Lifeboat(i, capacity=10) for i in range(1, 4)]
# fair_system = FairRescueSystem(fair_boats)
# fair_system.evacuate(demo_p2)
# fair_system.report_by_class()


---

## Розділ 16 — Фінальне резюме

### Таблиця концепцій

| Концепція | Що означає | Приклад у нашому коді |
|-----------|------------|----------------------|
| **Наслідування (is-a)** | Структура: підклас є різновидом батька | `ThirdClassPassenger` is-a `Passenger` |
| **MRO** | Лінійний алгоритм пошуку методів (C3) | `FirstClassPassenger → Passenger → object` |
| **`super()`** | Наступний клас у MRO — не обов'язково батько | Кооперативний `D.say()` |
| **Перевизначення** | Перший знайдений метод у MRO перемагає | `lifeboat_priority()` у підкласах |
| **`@classmethod`** | Фабричний конструктор з альтернативними аргументами | `Passenger.from_series(row)` |
| **Композиція (has-a)** | Об'єкт містить інші об'єкти і керує ними | `RescueSystem` has-a `[Lifeboat]` |
| **Оркестрація** | Системна поведінка — наслідування її не дає | `RescueSystem.evacuate()` |

---

### П'ять правил, які рятують архітектуру

```
1. Наслідуй, якщо X is-a Y — і це правда концептуально.
   Не наслідуй «для зручності».

2. Компонуй, якщо X has-a Y.
   Або якщо описуєш поведінку системи — не структуру об'єкта.

3. super() ≠ батько. super() = наступний у MRO.
   При множинному наслідуванні — завжди викликай super().

4. Оркестрацію будуй в окремому класі-координаторі.
   Класи самі по собі не утворюють систему.

5. Якщо система «зламана» — шукай відсутній координатор, а не поганий клас.
```

---

### Діаграма архітектурного вибору

```
Питання: «Як пов'язати клас A і клас B?»
         ┌──────────────────────────────────┐
         │  A є різновидом B?               │
         │  Dog is-a Animal? → Наслідування │
         └──────────────────────────────────┘
         ┌──────────────────────────────────┐
         │  A використовує B?               │
         │  Car has-a Engine? → Композиція  │
         └──────────────────────────────────┘
         ┌──────────────────────────────────────────┐
         │  A керує групою B?                       │
         │  RescueSystem керує [Passenger]?         │
         │  → Окремий координатор (Composition!)    │
         └──────────────────────────────────────────┘
```


---

## Архітектурний висновок

Цей урок — не просто про Python. Він про те, **як думати про системи**.

---

### Що дає наслідування

```
ThirdClassPassenger  is-a  Passenger
```

Наслідування відповідає на питання: **«Хто є хто?»**

Воно дає:
- повторне використання коду (не дублювати `is_child()` у кожному підкласі)
- спеціалізацію поведінки (різний `lifeboat_priority()` для кожного класу)
- семантику (`isinstance(jack, Passenger)` → True)

---

### Чого наслідування не дає

```
Наслідування не відповідає на питання:
  «Хто ким керує?»
  «В якому порядку відбуваються дії?»
  «Як об'єкти взаємодіють між собою?»
```

Це — **системна поведінка**. І вона потребує окремого компонента.

---

### Що дає композиція

```
RescueSystem  has-a  [Lifeboat]    ← знає про шлюпки
RescueSystem  has-a  [Passenger]   ← знає про пасажирів
RescueSystem.evacuate()            ← відповідає за поведінку системи
```

Композиція відповідає на питання: **«Як система поводиться?»**

---

### Формула реальних систем

```
Реальна система = структура + поведінка
                  (inheritance)  (composition + orchestration)

Тільки структура → набір ізольованих об'єктів, які нічим не керуються
Тільки поведінка → монолітний процедурний код без переваг ООП

Разом → гнучка, тестована, розширювана система
```

---

> **Головна думка:**  
> Класи описують, **що існує** у системі.  
> Координатор описує, **що відбувається** у системі.  
> Без координатора — це не система. Це просто добре описаний хаос.


---

## Розділ 17 — Питання для інтерв'ю

Ці питання реально задають на технічних інтерв'ю для Python-розробників:

---

**1. Що таке MRO і як Python його обчислює?**

> MRO (Method Resolution Order) — це впорядкований список класів, яким Python іде в пошуку методу чи атрибута. Python використовує алгоритм C3 лінеаризації, який гарантує, що підклас завжди передує батьківському, і що порядок батьківських класів зберігається. Переглянути MRO: `ClassName.mro()` або `ClassName.__mro__`.

---

**2. Що повертає `super()` — батьківський клас чи щось інше?**

> `super()` повертає проксі-об'єкт, що вказує на **наступний клас у MRO поточного об'єкта**. При одиночному наслідуванні це завжди батьківський клас. При множинному — може бути «сестринський» клас у ромбоподібній ієрархії.

---

**3. Що таке «diamond problem» і як Python його вирішує?**

> Diamond problem виникає коли клас `D` наслідує `B` і `C`, обидва з яких наслідують `A`. Без MRO метод `A` може бути викликаний двічі. Python вирішує це через C3 — клас `A` включається в MRO лише один раз, наприкінці.

---

**4. Коли краще використовувати композицію замість наслідування?**

> Композицію краще використовувати, коли:
- (1) відношення не є `is-a`, а `has-a`;
- (2) потрібна гнучка системна поведінка;
- (3) зв'язок між класами може змінюватись;
- (4) наслідування збільшує coupling.

> Принцип Gang of Four: «Favour composition over inheritance».

---

**5. Що таке `@classmethod` і чим він відрізняється від `@staticmethod`?**

> `@classmethod` отримує як перший аргумент сам клас (`cls`), а не екземпляр. Це дозволяє будувати альтернативні конструктори і фабричні методи. `@staticmethod` не отримує ні клас, ні екземпляр — це просто функція, прив'язана до namespace класу.

---

**6. Що означає `isinstance()` vs `type()` для перевірки типів?**

> `isinstance(obj, Passenger)` повертає `True` для всіх підкласів `Passenger` — воно перевіряє весь ланцюжок MRO. `type(obj) == Passenger` повертає `True` лише якщо об'єкт є *саме* `Passenger`, а не його підкласом. Для поліморфного коду завжди краще `isinstance()`.

---

**7. Що таке «оркестрація» і чому наслідування її не може забезпечити?**

> Оркестрація — це системна логіка, що координує взаємодію багатьох об'єктів. Наслідування описує структуру (хто ким є), але не може координувати поведінку різних об'єктів між собою. Для оркестрації потрібен окремий клас-координатор, побудований через композицію.


---

## Розділ 18 — Висновок

>У 1912 році на Titanic не вистачало шлюпок.
>
>Але навіть ті, що були, використовувались неефективно:
> - частина шлюпок відпливала напівпорожньою
> - не було єдиної системи координації
> - правила евакуації застосовувались непослідовно
>
> 👉 проблема була не лише в ресурсах, а й в організації процесу

---

Ми побудували дві системи:

**Наївна модель**  
- Пасажири мали пріоритет — але ніхто його не враховував  
- Місця займали ті, хто підійшов першими  
- Джек загинув не через поганий клас — через відсутній координатор  

**RescueSystem**  
- Координатор читав `lifeboat_priority()` кожного пасажира  
- Жінки та діти рятувались першими — як і задумувалось  
- Та сама кількість місць — інший результат  

---

### Що ви тепер знаєте

| # | Концепція | Суть |
|---|-----------|------|
| 1 | **Наслідування** | Описує структуру — хто ким є (`is-a`) |
| 2 | **MRO** | Лінійний ланцюжок пошуку методів (C3 алгоритм) |
| 3 | **`super()`** | Наступний у MRO — не завжди батько |
| 4 | **Перевизначення** | Перший знайдений метод у MRO виграє |
| 5 | **Множинне наслідування** | Потребує кооперативного `super()` скрізь |
| 6 | **Композиція** | Гнучкий `has-a` зв'язок через атрибути |
| 7 | **Оркестрація** | Системна логіка в окремому координаторі |

---

### Головна думка

```
Класи описують, ЩО є у системі.
Координатор описує, ЯК система поводиться.

Без координатора — це не система.
Це просто набір добре описаних об'єктів.
```

---

## 🧠 Важливий інсайт про MRO

MRO — це не проблема.

Це алгоритм, який гарантує:
- передбачуваність
- порядок викликів
- детермінізм

---

❗ Баги виникають не через MRO

А через:
- складні ієрархії
- неправильну модель
- відсутність координації системи

---

- 👉 MRO показує, ЯК працює система
- 👉 Але не відповідає за те, ЧИ правильно вона спроєктована
---

*Урок підготовлено в рамках курсу Python Intermediate, Module 2.*  
*Автор: Viktor Nikoriak — Spring 2026.*
